# A1.4 — Le nettoyage

On marque les spectres dont l'étiquette est douteuse (mauvais adduct, structure incohérente avec la masse).
Rien n'est supprimé : `meta.parquet` garde toutes les lignes et leur `row_id`, avec deux colonnes de filtre.

- **`keep`** : le spectre est fiable pour l'**entraînement** (toutes les règles) ;
- **`query_ok`** : le spectre peut servir de **requête de validation**. Seules comptent les règles qui
  n'utilisent que des informations présentes dans le test. Filtrer la validation avec la vraie structure la
  rendrait plus facile que le test.

Code : `src/casmi/clean.py`. Fichier : `python -m casmi.clean`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from casmi.clean import MAX_MASS_ERROR_DA, STRUCTURE_RULES, cleaning_rules, exact_masses, load_clean_meta, mass_error_da
from casmi.io import load_meta

pd.set_option("display.width", 200)

meta = load_meta(["inchikey14", "normalized_smiles", "ingest_lib", "ionization_mode", "adduct", "precursor_mz"])
exact = exact_masses(meta["normalized_smiles"])
rules = cleaning_rules(meta, exact)

## 1. Les règles

| Règle | Ce qu'elle repère | Utilise la structure ? |
|---|---|---|
| `adduct_inconnu` | adduct non interprétable : pas de masse neutre (`[Cat]2+`) | non |
| `charge_contre_mode` | adduct de charge opposée au mode d'ionisation (`[M-H]-` en mode positif) : physiquement impossible | non |
| `erreur_masse` | masse neutre déduite du précurseur à plus de 0,75 Da de la masse exacte du SMILES | **oui** |
| `adduct_douteux` | `[M-CH3]-` : précurseurs dispersés, qui ne correspondent à aucune formule | non |

### Pourquoi 0,75 Da ?

- Une erreur d'adduct décale la masse d'au moins un atome d'hydrogène (**1,008 Da**) : Na pris pour H fait
  21,98 Da, K pris pour H 37,96 Da.
- Un précurseur noté à l'unité près (appareils à basse résolution : 31 % des `precursor_mz` des quadripôles sont
  des entiers) s'écarte d'au plus **0,5 Da**.
- Les appareils à haute résolution se trompent de quelques ppm, soit environ 0,001 Da (Makarov *et al.*, 2006 ;
  Kind & Fiehn, 2006).

Un seuil en daltons, contrairement à un seuil en ppm, ne dépend pas de la masse de la molécule.

On ne se sert pas de `precursor_error_ppm` : B a montré qu'elle est décalée d'une masse d'électron, voire d'un
hydrogène, pour la plupart des adducts. L'écart est recalculé avec `compute_neutral_masses` et la masse
exacte RDKit.

In [ ]:
err = mass_error_da(meta, exact).abs()
bins = np.logspace(-5, 3, 81)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(err.clip(1e-5, 999), bins=bins,  # les écarts > 1 000 Da sont regroupés dans la dernière barre
        color="#2a78d6", edgecolor="white", linewidth=0.5)
ax.axvline(MAX_MASS_ERROR_DA, color="#3c3b37", linewidth=1.2, linestyle="--")
ax.annotate("seuil : 0,75 Da", (MAX_MASS_ERROR_DA, 0.92), xycoords=("data", "axes fraction"), xytext=(6, 0),
            textcoords="offset points", color="#3c3b37", fontsize=9)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|masse déduite du précurseur − masse exacte du SMILES| (Da, échelle log)")
ax.set_ylabel("spectres (échelle log)")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_title("Le seuil tombe dans le creux entre erreurs de mesure et erreurs d'étiquette", loc="left", fontsize=12)
plt.tight_layout()

À gauche, les erreurs de mesure : quelques millièmes de dalton pour les appareils précis, jusqu'à 0,5 Da pour
les précurseurs arrondis. À droite, à partir de 1 Da, les étiquettes fausses.

## 2. L'effet de chaque règle

Nombre de spectres repérés par chaque règle **seule**, puis **en plus** des règles précédentes (les règles se
chevauchent).

In [ ]:
already = pd.Series(False, index=rules.index)
rows = []
for name in rules.columns:
    rows.append({
        "règle": name,
        "seule": rules[name].sum(),
        "en plus des précédentes": (rules[name] & ~already).sum(),
        "molécules touchées": meta.loc[rules[name], "inchikey14"].nunique(),
    })
    already |= rules[name]
effect = pd.DataFrame(rows).set_index("règle")
effect.loc["total"] = [already.sum(), already.sum(), meta.loc[already, "inchikey14"].nunique()]
effect

## 3. L'effet par bibliothèque

Aucune règle ne doit vider une bibliothèque par accident.

In [ ]:
by_lib = rules.groupby(meta["ingest_lib"]).mean().mul(100)
by_lib["écartés (%)"] = rules.any(axis=1).groupby(meta["ingest_lib"]).mean().mul(100)
by_lib["spectres"] = meta.groupby("ingest_lib").size()
by_lib.round(2).sort_values("spectres", ascending=False)

Les plus touchées sont gnps (4,8 %) et riken (4,0 %), surtout par `erreur_masse`. Aucune bibliothèque n'est vidée,
et enveda-180, d'où vient le test, n'est pas touchée.

## 4. La règle de masse plutôt qu'une liste noire d'adducts

B avait signalé `[M-H5O3]+` (décalage constant de −1,418 Da). La règle de masse écarte les spectres faux de cet
adduct, mais garde ceux dont la masse est juste :

In [ ]:
x = meta["adduct"] == "[M-H5O3]+"
pd.DataFrame({
    "écart (Da)": mass_error_da(meta[x], exact[x]).round(3),
    "écarté": rules.loc[x, "erreur_masse"],
}).groupby("écarté")["écart (Da)"].describe()[["count", "min", "50%", "max"]]

`[M-CH3]-` reste dans une règle à part : ses écarts (−0,44 à −0,11 Da) sont sous le seuil, mais dispersés et sans
formule possible.

## 5. Le fichier `meta.parquet`

In [ ]:
clean = load_clean_meta()
print(f"Spectres : {len(clean):,}, row_id de 0 à {clean['row_id'].max():,}")
print(f"keep     : {clean['keep'].sum():,} ({clean['keep'].mean():.2%})")
print(f"query_ok : {clean['query_ok'].sum():,} ({clean['query_ok'].mean():.2%}) — règles sans structure : "
      f"{[r for r in rules.columns if r not in STRUCTURE_RULES]}")
clean.head()

## 6. Ce que `keep` ne fait pas

Les spectres de moins de 10 pics et les adducts dimères (`[2M+…]`) ne ressemblent pas au test (A1.3), mais
leur étiquette est juste : ils restent dans `keep`. Les retirer de l'entraînement est une expérience à mener en
A4.1. On les repère directement avec `num_peaks` et `adduct`.

**Usage :**
- entraînement : `meta[meta["keep"] & meta["inchikey14"].isin(molécules_train)]` ;
- requêtes de validation : `meta[meta["query_ok"] & …]`, **jamais** `keep`.